# Классификация видов деревьев на основе массивов точек LiDAR с применением нейронных сетей (RandomForest, PointNet++)

**Этапы работы программы:**
1. Предобработка сырых файлов `.las`
2. Выделение независимой тестовой выборки
3. Создание и обучение нейросетей RandomForest, PointNet++
4. Оценка моделей

In [1]:
import os
import glob
import random
import shutil
import numpy as np
import laspy
import torch
import torch.nn.functional as F
from torch.utils.data import random_split
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import PointNetConv, global_max_pool, fps, radius
from torch.nn import Sequential, Linear, ReLU, BatchNorm1d, Dropout
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
DATASET_DIR = "./"
HOLDOUT_DIR = "holdout_test_data"
HOLDOUT_COUNT = 4

NUM_POINTS = 2048
GROUND_THRESHOLD = 0.4

OUT_POINTS_FILE = "dataset_points.npy"
OUT_LABELS_FILE = "dataset_labels.npy"
OUT_CLASSES_FILE = "class_mapping.npy"

#### Общая предобработка данных
Предобработка файлов `.las` для всех моделей

In [3]:
def process_point_cloud(filepath):
    try:
        las = laspy.read(filepath)
        points = np.vstack((las.x, las.y, las.z)).transpose()
        if len(points) == 0: return None

        z_min = np.min(points[:, 2])
        mask = points[:, 2] > (z_min + GROUND_THRESHOLD)
        points = points[mask]
        if len(points) == 0: return None
        
        current_num_points = points.shape[0]
        if current_num_points >= NUM_POINTS:
            indices = np.random.choice(current_num_points, NUM_POINTS, replace=False)
        else:
            indices = np.random.choice(current_num_points, NUM_POINTS, replace=True)
        points = points[indices]

        centroid = np.mean(points, axis=0)
        points -= centroid
        max_distance = np.max(np.sqrt(np.sum(points**2, axis=1)))
        if max_distance > 0: points /= max_distance
        
        return points.astype(np.float32)
    except Exception as e:
        return None

# Запуск предобработки
os.makedirs(HOLDOUT_DIR, exist_ok=True)
class_names = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d)) and not d.startswith('.') and d != HOLDOUT_DIR]
class_names.sort() 
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

all_points, all_labels = [], []
total_train, total_holdout = 0, 0

for class_name in class_names:
    class_dir = os.path.join(DATASET_DIR, class_name)
    las_files = glob.glob(os.path.join(class_dir, "*.las")) + glob.glob(os.path.join(class_dir, "*.LAS"))
    
    random.seed(42)
    random.shuffle(las_files)
    
    actual_holdout = min(HOLDOUT_COUNT, len(las_files) // 2)
    holdout_files = las_files[:actual_holdout]
    training_files = las_files[actual_holdout:]
    
    class_holdout_dir = os.path.join(HOLDOUT_DIR, class_name)
    os.makedirs(class_holdout_dir, exist_ok=True)
    for h_file in holdout_files:
        shutil.copy(h_file, os.path.join(class_holdout_dir, os.path.basename(h_file)))
        
    total_holdout += len(holdout_files)
    for filepath in training_files:
        pts = process_point_cloud(filepath)
        if pts is not None:
            all_points.append(pts)
            all_labels.append(class_to_idx[class_name])
            total_train += 1

X = np.array(all_points, dtype=np.float32)
y = np.array(all_labels, dtype=np.int64)

np.save(OUT_POINTS_FILE, X)
np.save(OUT_LABELS_FILE, y)
np.save(OUT_CLASSES_FILE, class_names)

print(f"Использовано для обучения/валидации: {total_train} файлов.")
print(f"Отложено для слепого теста: {total_holdout} файлов.")

Использовано для обучения/валидации: 730 файлов.
Отложено для слепого теста: 24 файлов.


#### Визуализация предобработанных данных
Здесь можно посмотреть, как именно выглядят наши деревья после предобработки

In [4]:
raw_files = []

for class_name in os.listdir(DATASET_DIR):
    class_dir = os.path.join(DATASET_DIR, class_name)
    if os.path.isdir(class_dir) and class_name != HOLDOUT_DIR and not class_name.startswith('.'):
        files = glob.glob(os.path.join(class_dir, "*.las")) + glob.glob(os.path.join(class_dir, "*.LAS"))
        for f in files:
            raw_files.append((f"Train: {class_name} / {os.path.basename(f)}", f))

if os.path.exists(HOLDOUT_DIR):
    for class_name in os.listdir(HOLDOUT_DIR):
        class_dir = os.path.join(HOLDOUT_DIR, class_name)
        if os.path.isdir(class_dir):
            files = glob.glob(os.path.join(class_dir, "*.las")) + glob.glob(os.path.join(class_dir, "*.LAS"))
            for f in files:
                raw_files.append((f"Holdout: {class_name} / {os.path.basename(f)}", f))

available_files = list(set(raw_files))
available_files.sort()

def plot_4_views(filepath):
    points = process_point_cloud(filepath) 
    if points is None: return

    x, y, z = points[:, 0], points[:, 1], points[:, 2]

    max_range = np.array([x.max()-x.min(), y.max()-y.min(), z.max()-z.min()]).max() / 2.0
    mid_x = (x.max()+x.min()) * 0.5
    mid_y = (y.max()+y.min()) * 0.5
    mid_z = (z.max()+z.min()) * 0.5

    fig = plt.figure(figsize=(20, 6))
    fig.suptitle(f"Файл: {os.path.basename(filepath)} | Точек: 2048", fontsize=16, y=0.95)

    views = [
        (0, -90, "Спереди"),
        (0, 0, "Сбоку"),
        (90, -90, "Сверху"),
        (15, -45, "Изометрия")
    ]

    for i, (elev, azim, title) in enumerate(views):
        ax = fig.add_subplot(1, 4, i+1, projection='3d')
        
        ax.scatter(x, y, z, c=z, cmap='jet', s=6, alpha=0.9, marker='o', edgecolors='none')
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(title, fontsize=14, pad=-10)
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
        
        ax.axis('off')

    plt.subplots_adjust(wspace=0.05, top=1.0, bottom=0.0)
    plt.show()

dropdown = widgets.Dropdown(
    options=available_files,
    description='Файл:',
    layout={'width': '400px'}
)

out = widgets.interactive_output(plot_4_views, {'filepath': dropdown})
display(dropdown, out)

Dropdown(description='Файл:', layout=Layout(width='400px'), options=(('Holdout: Alder / alder_03.las', 'holdou…

Output()

# Классическое машинное обучение (Random Forest)
Эта часть кода обучает модель Random Forest

In [5]:
# Функция преобразования 3D-данных в таблицу признаков

def extract_features_from_cloud(points):
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    features = []
    
    # Габариты
    features.append(np.max(z) - np.min(z))
    features.append(np.max(x) - np.min(x))
    features.append(np.max(y) - np.min(y))
    
    # Дисперсия (разброс точек)
    features.append(np.var(x))
    features.append(np.var(y))
    features.append(np.var(z))
    
    # Перцентили высоты (форма кроны)
    features.extend(np.percentile(z, [10, 25, 50, 75, 90]))
    
    # Плотность по третям высоты
    z_min, z_max = np.min(z), np.max(z)
    h33 = z_min + (z_max - z_min) * 0.33
    h66 = z_min + (z_max - z_min) * 0.66
    features.extend([
        np.sum(z < h33) / len(z),
        np.sum((z >= h33) & (z < h66)) / len(z),
        np.sum(z >= h66) / len(z)
    ])
    
    # Главные компоненты (PCA)
    cov_matrix = np.cov(points, rowvar=False)
    eigenvalues, _ = np.linalg.eigh(cov_matrix)
    features.extend(np.sort(eigenvalues)[::-1])
    
    return np.array(features)

In [6]:
print("Загрузка нормализованных точек...")
X_raw = np.load(OUT_POINTS_FILE)
y_rf = np.load(OUT_LABELS_FILE)
class_names = np.load(OUT_CLASSES_FILE)

print("Извлечение геометрических признаков...")
X_features = np.array([extract_features_from_cloud(cloud) for cloud in X_raw])

# Разбитие на обучающую и тренировную выборки (80/20)
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_features, y_rf, test_size=0.2, random_state=42, stratify=y_rf
)

print(f"Обучение Random Forest (Train: {len(X_train_rf)}, Test: {len(X_test_rf)})...")
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train_rf, y_train_rf)

# Оценка модели
y_pred_rf = rf_model.predict(X_test_rf)
acc_rf = accuracy_score(y_test_rf, y_pred_rf)

print(f"Точность Random Forest: {acc_rf * 100:.2f}%")
print("\nДетальный отчет по классам:")
print(classification_report(y_test_rf, y_pred_rf, target_names=class_names))

Загрузка нормализованных точек...
Извлечение геометрических признаков...
Обучение Random Forest (Train: 584, Test: 146)...
Точность Random Forest: 82.88%

Детальный отчет по классам:
              precision    recall  f1-score   support

       Alder       0.89      0.80      0.84        10
       Aspen       0.85      0.79      0.81        28
       Birch       0.79      0.63      0.70        30
         Fir       0.75      0.86      0.80        14
        Pine       0.84      0.96      0.90        27
      Spruce       0.85      0.92      0.88        37

    accuracy                           0.83       146
   macro avg       0.83      0.83      0.82       146
weighted avg       0.83      0.83      0.82       146



#### Интерактивная проверка Random Forest
Предпросмотр работы RandomForest

In [7]:
import ipywidgets as widgets
from IPython.display import display, clear_output

raw_files = []

for class_name in os.listdir(DATASET_DIR):
    class_dir = os.path.join(DATASET_DIR, class_name)
    if os.path.isdir(class_dir) and class_name != HOLDOUT_DIR and not class_name.startswith('.'):
        files = glob.glob(os.path.join(class_dir, "*.las")) + glob.glob(os.path.join(class_dir, "*.LAS"))
        for f in files:
            raw_files.append((f"Train: {class_name} / {os.path.basename(f)}", f))

if os.path.exists(HOLDOUT_DIR):
    for class_name in os.listdir(HOLDOUT_DIR):
        class_dir = os.path.join(HOLDOUT_DIR, class_name)
        if os.path.isdir(class_dir):
            files = glob.glob(os.path.join(class_dir, "*.las")) + glob.glob(os.path.join(class_dir, "*.LAS"))
            for f in files:
                raw_files.append((f"Holdout: {class_name} / {os.path.basename(f)}", f))

available_files = list(set(raw_files))
available_files.sort()

def interactive_rf_prediction(filepath):
    if not filepath:
        return
        
    print(f"Обработка файла: {os.path.basename(filepath)}...")
    
    points = process_point_cloud(filepath)

    features = extract_features_from_cloud(points)
    features_2d = features.reshape(1, -1)
    
    pred_idx = rf_model.predict(features_2d)[0]
    probabilities = rf_model.predict_proba(features_2d)[0]
    
    predicted_class = class_names[pred_idx]
    confidence = probabilities[pred_idx] * 100
    
    print(" Результаты Random Forest")
    print(f"➤ Ответ модели:  {predicted_class.upper()}")
    print(f"➤ Уверенность:   {confidence:.1f}%\n")
    
    print("Вероятности :")
    top_indices = np.argsort(probabilities)[::-1]
    for i in range(min(5, len(class_names))):
        idx = top_indices[i]
        print(f"   {i+1}. {class_names[idx]:<10} - {probabilities[idx]*100:>5.1f}%")

dropdown_rf = widgets.Dropdown(
    options=available_files,
    description='Файл:',
    layout={'width': '400px'}
)

output_rf = widgets.Output()

def on_dropdown_change(change):
    with output_rf:
        clear_output(wait=True)
        interactive_rf_prediction(change.new)

dropdown_rf.observe(on_dropdown_change, names='value')
display(dropdown_rf, output_rf)

with output_rf:
    interactive_rf_prediction(dropdown_rf.value)

Dropdown(description='Файл:', layout=Layout(width='400px'), options=(('Holdout: Alder / alder_03.las', 'holdou…

Output()

# Глубокое машинное обучение (PointNet++)

Эта часть кода обучает модель PointNet++

In [8]:
BATCH_SIZE = 16
EPOCHS = 60
LEARNING_RATE = 0.001

MODEL_WEIGHTS = "best_pointnet_model.pth"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
FORCE_RETRAIN = True

class TreeDataset(torch.utils.data.Dataset):
    def __init__(self, p_file, l_file):
        self.points = torch.tensor(np.load(p_file), dtype=torch.float)
        self.labels = torch.tensor(np.load(l_file), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return Data(pos=self.points[idx], y=self.labels[idx])

class SetAbstraction(torch.nn.Module):
    def __init__(self, ratio, r, nn):
        super().__init__()
        self.ratio, self.r = ratio, r
        self.conv = PointNetConv(nn, add_self_loops=False)
    def forward(self, x, pos, batch):
        idx = fps(pos, batch, ratio=self.ratio)
        row, col = radius(pos, pos[idx], self.r, batch, batch[idx], max_num_neighbors=64)
        edge_index = torch.stack([col, row], dim=0)
        x_dst = None if x is None else x[idx]
        x = self.conv((x, x_dst), (pos, pos[idx]), edge_index)
        return x, pos[idx], batch[idx]

class PointNetPlusPlus(torch.nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.sa1 = SetAbstraction(0.5, 0.2, Sequential(Linear(3, 64), BatchNorm1d(64), ReLU(), Linear(64, 64), BatchNorm1d(64), ReLU(), Linear(64, 128), BatchNorm1d(128), ReLU()))
        self.sa2 = SetAbstraction(0.25, 0.4, Sequential(Linear(128 + 3, 128), BatchNorm1d(128), ReLU(), Linear(128, 128), BatchNorm1d(128), ReLU(), Linear(128, 256), BatchNorm1d(256), ReLU()))
        self.lin1, self.bn1 = Linear(256, 128), BatchNorm1d(128)
        self.drop = Dropout(0.5)
        self.lin2 = Linear(128, num_classes)
    def forward(self, data):
        pos, batch = data.pos, data.batch
        x, pos, batch = self.sa1(None, pos, batch)
        x, pos, batch = self.sa2(x, pos, batch)
        x = global_max_pool(x, batch)
        return self.lin2(self.drop(F.relu(self.bn1(self.lin1(x)))))

if os.path.exists(MODEL_WEIGHTS) and not FORCE_RETRAIN:
    print(f"Найден обученный файл: {MODEL_WEIGHTS}")
else:
    print("Запуск обучения PointNet++...")
    class_names = np.load(OUT_CLASSES_FILE)
    
    def train_step(model, loader, optimizer):
        model.train()
        loss_all, correct = 0, 0
        for data in loader:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = F.cross_entropy(out, data.y)
            loss.backward()
            optimizer.step()
            loss_all += loss.item() * data.num_graphs
            correct += int((out.argmax(dim=1) == data.y).sum())
        return loss_all / len(loader.dataset), correct / len(loader.dataset)

    @torch.no_grad()
    def test_step(model, loader):
        model.eval()
        correct = sum(int((model(data.to(device)).argmax(dim=1) == data.y).sum()) for data in loader)
        return correct / len(loader.dataset)

    full_dataset = TreeDataset(OUT_POINTS_FILE, OUT_LABELS_FILE)
    train_size = int(0.8 * len(full_dataset))
    train_dataset, val_dataset = random_split(full_dataset, [train_size, len(full_dataset) - train_size], generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    model = PointNetPlusPlus(num_classes=len(class_names)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_acc = 0
    for epoch in range(1, EPOCHS + 1):
        loss, train_acc = train_step(model, train_loader, optimizer)
        val_acc = test_step(model, val_loader)
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), MODEL_WEIGHTS)
            marker = "⭐ (Сохранено)"
        else: marker = ""
            
        print(f'Эпоха {epoch:03d} | Loss: {loss:.4f} | Train Acc: {train_acc*100:.1f}% | Val Acc: {val_acc*100:.1f}% {marker}')

    print(f"\nОбучение завершено! Лучшая точность: {best_acc*100:.2f}%.")

Запуск обучения PointNet++...
Эпоха 001 | Loss: 1.5362 | Train Acc: 36.5% | Val Acc: 19.2% ⭐ (Сохранено)
Эпоха 002 | Loss: 1.1174 | Train Acc: 56.2% | Val Acc: 30.1% ⭐ (Сохранено)
Эпоха 003 | Loss: 0.9805 | Train Acc: 63.2% | Val Acc: 61.0% ⭐ (Сохранено)
Эпоха 004 | Loss: 0.8313 | Train Acc: 68.3% | Val Acc: 70.5% ⭐ (Сохранено)
Эпоха 005 | Loss: 0.7504 | Train Acc: 74.8% | Val Acc: 63.0% 
Эпоха 006 | Loss: 0.6486 | Train Acc: 77.1% | Val Acc: 52.7% 
Эпоха 007 | Loss: 0.6197 | Train Acc: 78.1% | Val Acc: 71.2% ⭐ (Сохранено)
Эпоха 008 | Loss: 0.5409 | Train Acc: 82.0% | Val Acc: 67.8% 
Эпоха 009 | Loss: 0.4588 | Train Acc: 84.6% | Val Acc: 70.5% 
Эпоха 010 | Loss: 0.3640 | Train Acc: 89.0% | Val Acc: 71.2% 
Эпоха 011 | Loss: 0.4095 | Train Acc: 87.5% | Val Acc: 60.3% 
Эпоха 012 | Loss: 0.3576 | Train Acc: 89.2% | Val Acc: 47.3% 
Эпоха 013 | Loss: 0.3579 | Train Acc: 89.2% | Val Acc: 78.8% ⭐ (Сохранено)
Эпоха 014 | Loss: 0.3350 | Train Acc: 88.4% | Val Acc: 73.3% 
Эпоха 015 | Loss: 0.3260

#### Интерактивная проверка PointNet++
Предпросмотр работы PointNet++

In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from torch_geometric.data import Batch

class_names = np.load(OUT_CLASSES_FILE)
pn_model = PointNetPlusPlus(num_classes=len(class_names)).to(device)

try:
    pn_model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=device))
    pn_model.eval()
    model_loaded = True
except FileNotFoundError:
    print("Модель не обучена")
    model_loaded = False

def interactive_pn_prediction(filepath):
    if not filepath or not model_loaded:
        return
        
    print(f"Обработка файла: {os.path.basename(filepath)}...")
    points = process_point_cloud(filepath)
        
    # Превращаем numpy массив в тензор и оборачиваем в Batch
    pos_tensor = torch.tensor(points, dtype=torch.float).to(device)
    batch_data = Batch.from_data_list([Data(pos=pos_tensor)]).to(device)
    
    # Предсказание без расчёта градиентов
    with torch.no_grad():
        output = pn_model(batch_data)
        probabilities = F.softmax(output, dim=1)[0].cpu().numpy()
        
    pred_idx = np.argmax(probabilities)
    predicted_class = class_names[pred_idx]
    confidence = probabilities[pred_idx] * 100
    
    print(" Результаты PointNet++")
    print(f"➤ Ответ нейросети: {predicted_class.upper()}")
    print(f"➤ Уверенность:     {confidence:.1f}%\n")
    
    print("Вероятности (Топ-3):")
    top_indices = np.argsort(probabilities)[::-1]
    for i in range(min(3, len(class_names))):
        idx = top_indices[i]
        print(f"   {i+1}. {class_names[idx]:<10} - {probabilities[idx]*100:>5.1f}%")

# 3. Создание интерфейса
dropdown_pn = widgets.Dropdown(
    options=available_files, # Используем тот же список, что и раньше
    description='Файл:',
    layout={'width': '400px'}
)

output_pn = widgets.Output()

def on_dropdown_pn_change(change):
    with output_pn:
        clear_output(wait=True)
        interactive_pn_prediction(change.new)

dropdown_pn.observe(on_dropdown_pn_change, names='value')

if model_loaded:
    display(dropdown_pn, output_pn)
    # Запуск для первого файла по умолчанию
    with output_pn:
        interactive_pn_prediction(dropdown_pn.value)

Dropdown(description='Файл:', layout=Layout(width='400px'), options=(('Holdout: Alder / alder_03.las', 'holdou…

Output()